## Reprompt

In [1]:
from src.clone_gen import run_reprompt
import src.clone_gen as cg 
DeepSeek = "deepseek-r1:14b" 
Gemma3 = "gemma3:latest" 
Gpt20b= "gpt-oss:20b"
LLama3 = "llama3.1:latest" 
ALL_MODELS = [DeepSeek, Gemma3, LLama3, Gpt20b] 
DATASET_PATH = "../dataset/dataset.json"
OUT_PATH      = "../results/bigcodebench_llm_clones.json" 
FILTERED_PATH_CODEBLEU = "../results/bigcodebench_llm_clones_filtered_codebleu.json" 
FILTERED_PATH_REPROMPT = "../results/bigcodebench_llm_clones_filtered_reprompt.json" 
FILTERED_PATH_TESTS = "../results/bigcodebench_llm_clones_filtered_tests.json" 
FINAL_DATASET = "../results/bigcodebench_clone_dataset.json"  

# Generation settings
LLM_OPTS = {
    "temperature": 0.1,      # lower = more deterministic, higher = more creative
    "top_p": 0.95,            # nucleus sampling: consider only tokens with cumulative prob ≤ 0.95
    "repeat_penalty": 1.1,   # penalizes repeated tokens to avoid repetition
    "num_predict": 1500,       # max number of tokens the model will generate
}

from src.clone_gen import REMOTE_OLLAMA, N_ENTRIES 
cg.REMOTE_OLLAMA = REMOTE_OLLAMA

In [2]:
from src.clone_gen import call_ollama_chat
messages = [
    {"role": "user", "content": "Are you up and running, answer in one word."}
] 
response = call_ollama_chat(messages, DeepSeek, LLM_OPTS)
print(response)

Yes


In [3]:
run_reprompt(DATASET_PATH,FILTERED_PATH_CODEBLEU,N_ENTRIES,LLM_OPTS,ALL_MODELS,FILTERED_PATH_REPROMPT)


===== Processing entry 1/12 → BigCodeBench/58
🧠 Using model gemma3:latest (attempt 1)
🎯 CodeBLEU after reprompt: 0.2581
⚠️ Clone zero-shot deepseek-r1:14b-ast 1 ['refac_2', 'refac_5', 'refac_6'] still failing 5 tests after reprompt 1
🧠 Using model gpt-oss:20b (attempt 2)
🎯 CodeBLEU after reprompt: 0.3019
⚠️ Clone zero-shot deepseek-r1:14b-ast 1 ['refac_2', 'refac_5', 'refac_6'] still failing 5 tests after reprompt 2
🧠 Using model llama3.1:latest (attempt 3)
🎯 CodeBLEU after reprompt: 0.3019
⚠️ Clone zero-shot deepseek-r1:14b-ast 1 ['refac_2', 'refac_5', 'refac_6'] still failing 5 tests after reprompt 3
⚠️ Clone zero-shot deepseek-r1:14b-ast 1 ['refac_2', 'refac_5', 'refac_6'] still invalid after 3 attempts.
 Clone zero-shot deepseek-r1:14b-code 1 ['refac_1', 'refac_3', 'refac_4'] passes all tests — skipping reprompt.
🧠 Using model gemma3:latest (attempt 1)
🎯 CodeBLEU after reprompt: 0.3269
✅ Clone zero-shot deepseek-r1:14b-code 1 ['refac_2', 'refac_5', 'refac_6'] passes all tests afte

KeyboardInterrupt: 

## Filter based on tests

In [ ]:
import json

# --- Load both files ---
with open(FILTERED_PATH_CODEBLEU, "r", encoding="utf-8") as f:
    original_data = json.load(f)

with open(FILTERED_PATH_REPROMPT, "r", encoding="utf-8") as f:
    reprompt_data = json.load(f)

# --- Convert to dict keyed by entry ID (assuming each entry has a unique 'id') ---
def to_dict_by_id(data):
    return {entry["id"]: entry for entry in data}

orig_dict = to_dict_by_id(original_data)
reprompt_dict = to_dict_by_id(reprompt_data)

# --- Merge entries ---
merged_data = []

for entry_id in set(orig_dict.keys()) | set(reprompt_dict.keys()):
    # Prefer the reprompt entry if available
    base_entry = reprompt_dict.get(entry_id, orig_dict.get(entry_id)).copy()

    # Collect all clones from both (if both exist)
    orig_clones = {c["clone_id"]: c for c in orig_dict.get(entry_id, {}).get("clones", [])}
    reprompt_clones = {c["clone_id"]: c for c in reprompt_dict.get(entry_id, {}).get("clones", [])}

    # Merge, preferring reprompt versions
    merged_clones = {**orig_clones, **reprompt_clones}

    # Keep only passing clones
    passing_clones = [
        clone.copy()
        for clone in merged_clones.values()
        if clone.get("test_results") and all(r == "PASS" for r in clone["test_results"].values())
    ]

    if passing_clones:
        base_entry["clones"] = passing_clones
        merged_data.append(base_entry)

# --- Save filtered dataset ---
with open(FILTERED_PATH_TESTS, "w", encoding="utf-8") as f:
    json.dump(merged_data, f, indent=2, ensure_ascii=False)

print(f"✅ Filtered dataset saved to {FILTERED_PATH_TESTS}, entries: {len(merged_data)}")


✅ Filtered dataset saved to ../results/bigcodebench_llm_clones_filtered_tests.json, entries: 10


## Codebleu between all clones 

In [ ]:
import json
from codebleu import calc_codebleu
from itertools import combinations

with open(FILTERED_PATH_TESTS, "r", encoding="utf-8") as f:
    clone_data = json.load(f)

for i, clone_entry in enumerate(clone_data, 1):
    entry_id = clone_entry["id"]
    print(f"\nEvaluating Entry {i}/{len(clone_data)} | id={entry_id}")

    clones = clone_entry.get("clones", [])
    n = len(clones)

    for idx1, idx2 in combinations(range(n), 2):
        clone1 = clones[idx1]
        clone2 = clones[idx2]

        clone1_id = clone1.get("clone_id", f"clone_{idx1 + 1}")
        clone2_id = clone2.get("clone_id", f"clone_{idx2 + 1}")

        clone1.setdefault("metrics", {}).setdefault("codebleu", {})
        clone2.setdefault("metrics", {}).setdefault("codebleu", {})

        # Skip if already computed in either direction
        if clone2_id in clone1["metrics"]["codebleu"] and clone1_id in clone2["metrics"]["codebleu"]:
            print(f"  Skipping Clone {idx1 + 1} vs Clone {idx2 + 1} (already computed)")
            continue

        try:
            score = calc_codebleu([clone1["code"]], [clone2["code"]], lang="python")
            val = float(score["codebleu"])
            clone1["metrics"]["codebleu"][clone2_id] = val
            clone2["metrics"]["codebleu"][clone1_id] = val

            print(f"  Clone {idx1 + 1} vs Clone {idx2 + 1}: CodeBLEU={val:.4f}")
        except Exception as e:
            print(f"  ❌ Error computing CodeBLEU for clones {idx1 + 1} vs {idx2 + 1}: {e}")

    # Save after finishing each entry
    with open(FILTERED_PATH_TESTS, "w", encoding="utf-8") as f:
        json.dump(clone_data, f, indent=2)

print(f"\n✅ Done. Final dataset with CodeBLEU scores (clone vs clone) saved to {FILTERED_PATH_TESTS}")



Evaluating Entry 1/10 | id=BigCodeBench/58
  Clone 1 vs Clone 2: CodeBLEU=0.3436

Evaluating Entry 2/10 | id=BigCodeBench/151
  Clone 1 vs Clone 2: CodeBLEU=0.5002
  Clone 1 vs Clone 3: CodeBLEU=0.6380
  Clone 1 vs Clone 4: CodeBLEU=0.6348
  Clone 1 vs Clone 5: CodeBLEU=0.7811
  Clone 1 vs Clone 6: CodeBLEU=0.6184
  Clone 1 vs Clone 7: CodeBLEU=0.4949
  Clone 1 vs Clone 8: CodeBLEU=0.7057
  Clone 1 vs Clone 9: CodeBLEU=0.5189
  Clone 1 vs Clone 10: CodeBLEU=0.8282
  Clone 1 vs Clone 11: CodeBLEU=0.5415
  Clone 2 vs Clone 3: CodeBLEU=0.5770
  Clone 2 vs Clone 4: CodeBLEU=0.4683
  Clone 2 vs Clone 5: CodeBLEU=0.4811
  Clone 2 vs Clone 6: CodeBLEU=0.4582
  Clone 2 vs Clone 7: CodeBLEU=0.4574
  Clone 2 vs Clone 8: CodeBLEU=0.4586
  Clone 2 vs Clone 9: CodeBLEU=0.7449
  Clone 2 vs Clone 10: CodeBLEU=0.4654
  Clone 2 vs Clone 11: CodeBLEU=0.4683
  Clone 3 vs Clone 4: CodeBLEU=0.6852
  Clone 3 vs Clone 5: CodeBLEU=0.6003
  Clone 3 vs Clone 6: CodeBLEU=0.6439
  Clone 3 vs Clone 7: CodeBLEU=0.